# 05 — Analysis

Load the trained checkpoint, evaluate per-state prediction accuracy,
and compare the model's learned transition probabilities to the true
Gambler's Ruin matrix.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

DATA_DIR = Path("projects/markov-transformer/experiments/markov-chain-learning/data")

## Load model & data

In [ ]:
# Model definition (self-contained copy)
class MarkovTransformer(nn.Module):
    def __init__(
        self,
        vocab_size=5,
        d_model=32,
        nhead=4,
        num_layers=2,
        dim_feedforward=64,
        max_len=64,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=0.0,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L = x.shape
        positions = torch.arange(L, device=x.device)
        h = self.token_embedding(x) + self.pos_embedding(positions)
        mask = torch.triu(
            torch.full((L, L), float("-inf"), device=x.device), diagonal=1
        )
        h = self.transformer(h, mask=mask, is_causal=True)
        return self.head(h)

In [ ]:
ckpt = torch.load(DATA_DIR / "checkpoint.pt", weights_only=True)
cfg = ckpt["config"]

model = MarkovTransformer(**cfg)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(
    f"Loaded checkpoint from epoch {ckpt['epoch']}  "
    f"(val loss {ckpt['best_val_loss']:.4f})"
)

data = torch.load(DATA_DIR / "sequences.pt", weights_only=True)
T_true = data["T"]
seqs = data["sequences"]
print(f"True T:\n{T_true}")

## Per-state next-token accuracy

In [ ]:
# For each position t in each sequence, check whether argmax(logits[t]) == seq[t+1].
# Break out accuracy by the *current* state (seq[t]) to get a per-state view.

VOCAB = 5
correct = torch.zeros(VOCAB)
totals = torch.zeros(VOCAB)

BATCH = 512
with torch.no_grad():
    for start in range(0, len(seqs), BATCH):
        batch = seqs[start : start + BATCH]  # (B, L)
        x_in = batch[:, :-1]  # (B, L-1)
        x_tgt = batch[:, 1:]  # (B, L-1)
        logits = model(x_in)  # (B, L-1, V)
        preds = logits.argmax(dim=-1)  # (B, L-1)

        for state in range(VOCAB):
            mask = x_in == state
            totals[state] += mask.sum()
            correct[state] += (preds[mask] == x_tgt[mask]).sum()

acc = (correct / totals.clamp(min=1)).numpy()
print("Per-state accuracy:")
for s in range(VOCAB):
    bar = "#" * int(acc[s] * 40)
    print(f"  State {s}: {acc[s]:.1%}  {bar}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(VOCAB), acc)
ax.set_xticks(range(VOCAB))
ax.set_xlabel("Current state")
ax.set_ylabel("Next-token accuracy")
ax.set_title("Per-state next-token prediction accuracy")
ax.set_ylim(0, 1)
ax.axhline(0.6, color="gray", linestyle="--", linewidth=0.8, label="p=0.6")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Learned vs true transition matrix

Feed each state as a single-token context; the model's softmax output at
position 0 approximates T[state, :] (the next-state distribution given
a one-step history).

In [ ]:
# Single-token context: shape (5, 1)
all_states = torch.arange(VOCAB).unsqueeze(1)
with torch.no_grad():
    logits_single = model(all_states)  # (5, 1, 5)
    T_learned = logits_single[:, 0, :].softmax(dim=-1)  # (5, 5)

print("Learned T (single-step context):")
print(T_learned.numpy().round(3))
print("\nTrue T:")
print(T_true.numpy())
print("\nMax absolute error:", (T_learned - T_true).abs().max().item())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

kw = dict(
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    linewidths=0.3,
    xticklabels=[f"→{j}" for j in range(VOCAB)],
    yticklabels=[str(i) for i in range(VOCAB)],
)

sns.heatmap(T_true.numpy(), ax=axes[0], **kw)
axes[0].set_title("True T")

sns.heatmap(T_learned.detach().numpy(), ax=axes[1], **kw)
axes[1].set_title("Learned T (single-step)")

diff = (T_learned - T_true).detach().numpy()
sns.heatmap(
    diff,
    ax=axes[2],
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    xticklabels=[f"→{j}" for j in range(VOCAB)],
    yticklabels=[str(i) for i in range(VOCAB)],
)
axes[2].set_title("Difference  (learned − true)")

for ax in axes:
    ax.set_xlabel("Next state")
    ax.set_ylabel("Current state")

plt.suptitle(
    "Transformer learned transition probabilities vs ground truth", fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()

## Multi-step context: does more history help?

The Markov property implies that *only* the current state matters.
Feed longer prefixes ending in each state and check whether the full-context
predictions converge to the true T row.

In [ ]:
from markov_chain import generate_sequences

T_true_local = T_true

# For each state i, collect sequences that are at state i at their last position,
# then compare the model's predicted distribution to T[i, :].
CONTEXT_LEN = 8
test_seqs = generate_sequences(
    T_true_local, num_sequences=5_000, seq_len=CONTEXT_LEN + 1, seed=99
)

T_ctx = torch.zeros(VOCAB, VOCAB)
counts_ctx = torch.zeros(VOCAB)

with torch.no_grad():
    logits_ctx = model(test_seqs[:, :-1])  # (N, CONTEXT_LEN, V)
    probs_ctx = logits_ctx[:, -1, :].softmax(dim=-1)  # (N, V)

for state in range(VOCAB):
    mask = test_seqs[:, -2] == state  # sequences where pos-before-last == state
    if mask.sum() == 0:
        continue
    T_ctx[state] = probs_ctx[mask].mean(dim=0)
    counts_ctx[state] = mask.sum()

print(f"Avg predicted T (context len {CONTEXT_LEN}):")
print(T_ctx.numpy().round(3))
print("Max abs error:", (T_ctx - T_true_local).abs().max().item())